# Import

In [ ]:
!pip install torchsummaryX wandb --quiet

import torch
import torch.nn as nn
import numpy as np
from torchsummaryX import summary
import sklearn
import sklearn.metrics
import gc
import pandas as pd
import os
import wandb
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


# Acquire the Dataset

In [ ]:
!pip install --upgrade kaggle
!mkdir /root/.kaggle

with open("/root/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"skyprotector","key":"9783f55e9344bbb8e4f86a3d29936e69"}')

!chmod 600 /root/.kaggle/kaggle.json

!kaggle competitions download -c spaceship-titanic
!unzip spaceship-titanic.zip

mkdir: cannot create directory ‘/root/.kaggle’: File exists
spaceship-titanic.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  spaceship-titanic.zip
replace sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: sample_submission.csv   
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test.csv                
replace train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train.csv               


# Examine Dataset

In [ ]:
train_data = pd.read_csv("/content/train.csv")
test_data = pd.read_csv("/content/test.csv")
#test_data.drop(['PassengerId','Cabin','Destination','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','Name'], axis = 1, inplace=True)
#test_data
#test_data.to_csv("/content/test.csv",index=False)
#train_data.head()
#print(type(train_data["PassengerId"]))
#test_data = pd.read_csv("/content/test.csv")
#test_data.head()

# Val

In [ ]:
train_data = pd.read_csv("/content/train.csv")
num_items = len(train_data)
split_ind = num_items // 5*4

val_data = train_data.iloc[split_ind:num_items]
train_data = train_data.iloc[0:split_ind]

train_data.to_csv("/content/train.csv",index=False)
val_data.to_csv("/content/val.csv",index=False)

# Dataset class

In [ ]:
class TitanicDataset(torch.utils.data.Dataset):

    def __init__(self, data_path, partition="train"):

        self.data_path = data_path
        self.partition = partition

        # Normally datasets are not as convienient as a single csv, in which case we usually build a for-loop and iterate over files;
        raw_data = pd.read_csv(self.data_path + "/" + self.partition + ".csv")

        # Preprocess the dataset here using whatever methods; here we only replace NaN with -1 for all columns
        self.full_data = raw_data.fillna(-1)
        #transported_map = {"False": 0, "True": 1}
        #self.full_data = self.full_data.replace({"Transported": transported_map})
        self.full_data['Transported'].replace({False: 0, True: 1}, inplace=True)
        vip_map = {"False": 0, "True": 1}
        self.full_data = self.full_data.replace({"VIP": vip_map})
        sleep_map = {"False": 0, "True": 1}
        self.full_data = self.full_data.replace({"CryoSleep": sleep_map})
        planet_map = {"Earth": 1, "Europa": 2, "Mars": 3, "not listed": 4}
        self.full_data = self.full_data.replace({"HomePlanet": planet_map})
        self.length = len(self.full_data)

    def __len__(self):

        return self.length

    def __getitem__(self, ind):

        entry = self.full_data.iloc[ind]

        transported = entry.loc["Transported"]
        # id = entry.loc["PassengerId"]
        planet = entry.loc["HomePlanet"]
        age = entry.loc["Age"]
        vip = entry.loc["VIP"]
        sleep = entry.loc["CryoSleep"]
        inputs = torch.FloatTensor([planet, age, vip, sleep])
        survived = torch.FloatTensor([transported])

        return survived, inputs

In [ ]:
class TitanicTestDataset(torch.utils.data.Dataset):

    def __init__(self, data_path):

        self.data_path = data_path

        # Normally datasets are not as convienient as a single csv, in which case we usually build a for-loop and iterate over files;
        raw_data = pd.read_csv(self.data_path + "/test.csv")

        # Preprocess the dataset here using whatever methods; here we only replace NaN with -1 for all columns
        self.full_data = raw_data.fillna(-1)
        #transported_map = {"False": 0, "True": 1}
        #self.full_data = self.full_data.replace({"Transported": transported_map})
        #self.full_data['Transported'].replace({False: 0, True: 1}, inplace=True)
        vip_map = {"False": 0, "True": 1}
        self.full_data = self.full_data.replace({"VIP": vip_map})
        sleep_map = {"False": 0, "True": 1}
        self.full_data = self.full_data.replace({"CryoSleep": sleep_map})
        planet_map = {"Earth": 1, "Europa": 2, "Mars": 3, "not listed": 4}
        self.full_data = self.full_data.replace({"HomePlanet": planet_map})
        self.length = len(self.full_data)

    def __len__(self):

        return self.length

    def __getitem__(self, ind):

        entry = self.full_data.iloc[ind]

        #id = entry.loc["PassengerId"]
        planet = entry.loc["HomePlanet"]
        age = entry.loc["Age"]
        vip = entry.loc["VIP"]
        sleep = entry.loc["CryoSleep"]
        inputs = torch.FloatTensor([planet, age, vip, sleep])
        return inputs



# Macros

In [ ]:
config = {
    'epochs': 15,
    'batch_size' : 16,
    'learning_rate' : 0.001,
    'architecture' : 'tmp',
    'dropout' : 0.1,
    'step_size' : 3,
    'gamma' : 0.05,

}

# Load

In [ ]:
content_path = '/content'

train_data = TitanicDataset(content_path, "train")
val_data = TitanicDataset(content_path, "val")
test_data = TitanicTestDataset(content_path)
train_loader = torch.utils.data.DataLoader(train_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= True)
val_loader = torch.utils.data.DataLoader(val_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= False)
test_loader = torch.utils.data.DataLoader(test_data, num_workers= 2,
                                           batch_size=config['batch_size'], pin_memory= True,
                                           shuffle= False)

print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

Train dataset samples = 6952, batches = 435
Val dataset samples = 1741, batches = 109
Test dataset samples = 4277, batches = 268


In [ ]:
for i, data in enumerate(train_loader):
    targets, inputs = data
    print(targets.shape, inputs.shape)
    break
for i, data in enumerate(test_loader):
    inputs = data
    print(inputs.shape)
    break

torch.Size([16, 1]) torch.Size([16, 4])
torch.Size([16, 4])


# Network

In [ ]:
class Network(nn.Module):

    def __init__(self, input_size, dropout):

        super(Network, self).__init__()

        output_size = 1

        self.model = nn.Sequential(

            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, output_size),

        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.model(x)
        out = self.sigmoid(out)

        return out

# Init model

In [ ]:
targets, inputs = next(iter(train_loader))

# Model init here
model = Network(inputs.shape[1], config['dropout']).to(device)

# Check your model architecture and parameters
summary(model, inputs.to(device))

                  Kernel Shape Output Shape Params Mult-Adds
Layer                                                       
0_model.Linear_0      [4, 128]    [16, 128]  640.0     512.0
1_model.ReLU_1               -    [16, 128]      -         -
2_model.Dropout_2            -    [16, 128]      -         -
3_model.Linear_3      [128, 1]      [16, 1]  129.0     128.0
4_sigmoid                    -      [16, 1]      -         -
--------------------------------------------------------------
                      Totals
Total params           769.0
Trainable params       769.0
Non-trainable params     0.0
Mult-Adds              640.0


/usr/local/lib/python3.10/dist-packages/torchsummaryX/torchsummaryX.py:101: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  df_sum = df.sum()


,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_model.Linear_0,"[4, 128]","[16, 128]",640.0,512.0
1_model.ReLU_1,-,"[16, 128]",NaN,NaN
2_model.Dropout_2,-,"[16, 128]",NaN,NaN
3_model.Linear_3,"[128, 1]","[16, 1]",129.0,128.0
4_sigmoid,-,"[16, 1]",NaN,NaN


# Loss

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, config['step_size'], config['gamma'])

# Empty CUDA Cache & Reclaim Memory

In [ ]:
torch.cuda.empty_cache()
gc.collect()

1448

# Train

In [ ]:
def train(model, optimizer, criterion, dataloader, scaler):

    model.train()
    train_loss = 0.0

    for iter, (targets, inputs) in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        targets = targets.to(device)
        inputs = inputs.to(device)

        # Forward Propagation
        outputs = model(inputs)

        # Loss Calculation
        loss = criterion(outputs, targets)
        train_loss += loss.item()

        # Initialize Gradients
        optimizer.zero_grad()

        # Backward Propagation
        scaler.scale(loss).backward()

        # Gradient Descent
        scaler.step(optimizer)

        scaler.update()

    train_loss /= len(dataloader)
    return train_loss

# Eval

In [ ]:
def eval(model, dataloader):

    model.eval() # Set model in evaluation mode

    ground_truth_list = []
    outputs_list = []

    for i, (targets, inputs) in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        targets = targets.to(device)
        inputs = inputs.to(device)

        with torch.inference_mode(): # Make sure that there are no gradients computed as we are not training the model now
            # Forward Propagation
            outputs = model(inputs)

        # Store GT & Outputs
        ground_truth_list.extend(targets.cpu().tolist())
        outputs_list.extend(outputs.round().cpu().tolist())

    # Calculate Accuracy
    accuracy = sklearn.metrics.accuracy_score(outputs_list, ground_truth_list)
    return accuracy*100

# W&B

In [ ]:
wandb.login(key="5da5d90f8b775bd6ff70ae7fbd6d006118133cd6") # API Key is in your wandb account, under settings (wandb.ai/settings)

# Create your wandb run
run = wandb.init(
    name = "spacetitanic",
    reinit = True,
    project = "bootcamp", # Project should be created in your wandb account
    config = config
)

# Save your model architecture as a string with str(model)
model_arch = str(model)

# Save it in a txt file
arch_file = open("model_arch.txt", "w")
file_write = arch_file.write(model_arch)
arch_file.close()

# log it in your wandb run with wandb.save()
wandb.save('model_arch.txt')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


['/content/wandb/run-20231220_180737-pzmkjtgs/files/model_arch.txt']

# Experiement

In [ ]:
best_acc = 0.0

scaler = torch.cuda.amp.GradScaler()

for epoch in range(config['epochs']):
    print("\nEpoch {}/{}".format(epoch+1, config['epochs']))

    train_loss = train(model, optimizer, criterion, train_loader, scaler)
    accuracy = eval(model, val_loader)

    print("\tTrain Loss: {:.4f}".format(train_loss))
    print("\tValidation Accuracy: {:.2f}%".format(accuracy))

    # Log metrics at each epoch in your run
    # wandb.log({"train loss": train_loss, "validation accuracy": accuracy})

    # Save checkpoint if accuracy is better than your current best
    if accuracy >= best_acc:

      # Save checkpoint with information you want
      torch.save({'epoch': epoch,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'loss': train_loss,
              'acc': accuracy},
        './model_checkpoint.pth')

      # Save checkpoint in wandb
      # wandb.save('checkpoint.pth')

    scheduler.step()


Epoch 1/15
	Train Loss: 0.8046
	Validation Accuracy: 73.52%

Epoch 2/15
	Train Loss: 0.6520
	Validation Accuracy: 75.42%

Epoch 3/15
	Train Loss: 0.6145
	Validation Accuracy: 74.84%

Epoch 4/15
	Train Loss: 0.5906
	Validation Accuracy: 75.07%

Epoch 5/15
	Train Loss: 0.5872
	Validation Accuracy: 75.19%

Epoch 6/15
	Train Loss: 0.5858
	Validation Accuracy: 75.19%

Epoch 7/15
	Train Loss: 0.5859
	Validation Accuracy: 75.19%

Epoch 8/15
	Train Loss: 0.5861
	Validation Accuracy: 75.19%

Epoch 9/15
	Train Loss: 0.5850
	Validation Accuracy: 75.19%

Epoch 10/15
	Train Loss: 0.5841
	Validation Accuracy: 75.19%

Epoch 11/15
	Train Loss: 0.5852
	Validation Accuracy: 75.19%

Epoch 12/15
	Train Loss: 0.5848
	Validation Accuracy: 75.19%

Epoch 13/15
	Train Loss: 0.5864
	Validation Accuracy: 75.19%

Epoch 14/15
	Train Loss: 0.5848
	Validation Accuracy: 75.19%

Epoch 15/15
	Train Loss: 0.5817
	Validation Accuracy: 75.19%


# Testing

In [ ]:
def test(model, dataloader):

    model.eval()

    outputs_list = []

    for i, inputs in enumerate(dataloader):

        # Move Data to Device (Ideally GPU)
        inputs = inputs.to(device)

        with torch.inference_mode(): # Make sure that there are no gradients computed as we are not training the model now
            # Forward Propagation
            outputs = model(inputs)

        # Store Outputs
        outputs_list.extend(outputs.round().type(torch.int64).cpu().tolist())

    return sum(outputs_list, [])

In [ ]:
test_data = test_data = pd.read_csv("/content/test.csv")
predictions = test(model, test_loader)
test_id = list(test_data['PassengerId'])
# Create CSV file with predictions
with open(content_path + "/submission.csv", "w+") as f:
    f.write("PassengerId,Transported\n")
    for i in range(len(predictions)):
        f.write("{},{}\n".format(test_id[i],bool(predictions[i])))

# Victory!!!!!

In [ ]:
!kaggle competitions submit -c spaceship-titanic -f submission.csv -m "Message"

100% 56.9k/56.9k [00:01<00:00, 58.1kB/s]
Successfully submitted to Spaceship Titanic